In [1]:
#@title MASTER — Persistent OpenVLA + Arm Agent + LIBERO + SimplerEnv Setup
# =============================================================================
# IDEMPOTENT / PERSISTENT COLAB SETUP
#
# Design:
#   1. Mount Google Drive
#   2. KEEP an existing /content/openvla_env
#   3. Otherwise restore it from Drive
#   4. If no archive exists, create Python 3.11 environment
#   5. KEEP existing repositories
#   6. Otherwise restore repository archive / clone
#   7. Install only missing dependencies
#   8. Repair only known compatibility pins when necessary
#   9. Check MuJoCo 3.3.1 — DO NOT reinstall if already correct
#  10. Fix LIBERO package marker only if missing
#  11. Copy only missing LIBERO BDDL files
#  12. Verify CUDA / PyTorch / OpenVLA / LIBERO / SimplerEnv / Arm Agent
#  13. Actually construct a LIBERO OffScreenRenderEnv
#  14. Refresh Drive archives ONLY if this setup changed something
#
# IMPORTANT:
#   The current known-good inference environment uses:
#
#       Python             3.11
#       PyTorch            2.11.0+cu128
#       CUDA runtime       12.8
#       NumPy              1.26.4
#       MuJoCo             3.3.1
#       Flash Attention    2.5.5
#       TensorFlow         2.15.0
#       TFDS               4.9.3
#       TF Metadata        1.15.0
#       protobuf           4.25.9
#       huggingface-hub    0.24.7
#       setuptools         69.5.1
#
# =============================================================================

import os
import sys
import subprocess
import shutil
from pathlib import Path


# =============================================================================
# 0. CONFIGURATION
# =============================================================================

ENV = "/content/openvla_env"
MINICONDA = "/content/miniconda"
OPENVLA = "/content/openvla"
LIBERO = "/content/LIBERO"
ARMAGENT = "/content/panda_pick"
SIMPLER = "/content/SimplerEnv"

DRIVE_ROOT = "/content/drive/MyDrive/OpenVLA"

ENV_ARCHIVE = f"{DRIVE_ROOT}/openvla_env.tar.zst"
REPO_ARCHIVE = f"{DRIVE_ROOT}/armagent_repos.tar.zst"
HF_CACHE = f"{DRIVE_ROOT}/huggingface_cache"

PYTHON = f"{ENV}/bin/python"
CONDA = f"{MINICONDA}/bin/conda"

EXPECTED_PYTHON_MAJOR = 3
EXPECTED_PYTHON_MINOR = 11

# Known-good versions
NUMPY_VERSION = "1.26.4"
PROTOBUF_VERSION = "4.25.9"
TF_VERSION = "2.15.0"
TFDS_VERSION = "4.9.3"
TFMD_VERSION = "1.15.0"
HF_HUB_VERSION = "0.24.7"
SETUPTOOLS_VERSION = "69.5.1"
MUJOCO_VERSION = "3.3.1"
FLASH_ATTN_VERSION = "2.5.5"
OPENCV_VERSION = "4.10.0.84"
GYMNASIUM_VERSION = "0.29.1"

# Current known-good PyTorch.
# We VERIFY this rather than automatically changing it.
EXPECTED_TORCH_VERSION = "2.11.0"
EXPECTED_TORCH_CUDA = "12.8"

# If True, archives are rebuilt after every successful setup.
# Keep False for normal sessions.
FORCE_REFRESH_ARCHIVES = False

# Set True only if you intentionally want setup to rebuild the archives
# after verification.
REFRESH_ARCHIVES_IF_CHANGED = True

DO_ALL_CHECKS = False

# =============================================================================
# GLOBAL ENVIRONMENT FOR SUBPROCESSES
# =============================================================================

VERIFY_ENV = {
    **os.environ,

    # LIBERO package discovery
    "PYTHONPATH": f"{LIBERO}:{ARMAGENT}:{OPENVLA}",

    # Headless rendering
    "MPLBACKEND": "Agg",
    "MUJOCO_GL": "egl",

    # Persistent Hugging Face cache
    "HF_HOME": HF_CACHE,
    "HUGGINGFACE_HUB_CACHE": HF_CACHE,

    # Avoid tokenizer multiprocessing noise
    "TOKENIZERS_PARALLELISM": "false",
}



# =============================================================================
# HELPERS
# =============================================================================

changed = False


def mark_changed():
    print("marking as changed...")
    global changed
    changed = True


def run(cmd, cwd=None, env=None, input_data=None, timeout=600, check=True):
    """Run command and print complete output."""

    print("\n$ " + " ".join(str(x) for x in cmd))
    print("-" * 72)

    r = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        input=input_data,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )

    if r.stdout:
        print(r.stdout)

    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {r.returncode}:\n"
            + " ".join(str(x) for x in cmd)
        )

    return r


def package_version(package):
    """Return installed package version from the OpenVLA Python environment."""

    code = f"""
import importlib.metadata as md

try:
    print(md.version({package!r}))
except md.PackageNotFoundError:
    print("__NOT_INSTALLED__")
"""

    r = subprocess.run(
        [PYTHON, "-c", code],
        env=VERIFY_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=30,
    )

    if r.returncode != 0:
        return None

    value = r.stdout.strip().splitlines()[-1]

    if value == "__NOT_INSTALLED__":
        return None

    return value


def ensure_version(package, required_version, no_deps=True):
    """
    Install/change ONLY this package when necessary.

    Correct version:
        no action

    Missing:
        install

    Wrong:
        change only this package
    """

    current = package_version(package)

    if current == required_version:
        print(f"✓ {package}=={required_version} already installed — SKIP")
        return False

    if current is None:
        print(f"→ {package} missing — installing {required_version}")
    else:
        print(
            f"→ {package}=={current} found; "
            f"required {required_version} — changing ONLY this package"
        )

    cmd = [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"{package}=={required_version}",
    ]

    if no_deps:
        cmd.insert(4, "--no-deps")

    run(cmd, timeout=600)

    actual = package_version(package)

    if actual != required_version:
        raise RuntimeError(
            f"{package}: expected {required_version}, found {actual}"
        )

    mark_changed()
    print(f"✓ {package}=={actual}")

    return True


def import_ok(module):
    r = subprocess.run(
        [
            PYTHON,
            "-c",
            f"import {module}",
        ],
        env=VERIFY_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=60,
    )

    return r.returncode == 0


def install_if_missing(package, module=None, extra_args=None):
    """
    Install only when the import is missing.
    """

    module = module or package

    if import_ok(module):
        print(f"✓ {package}: import already works — SKIP")
        return False

    print(f"→ {package}: missing — installing")

    cmd = [
        PYTHON,
        "-m",
        "pip",
        "install",
    ]

    if extra_args:
        cmd.extend(extra_args)

    cmd.append(package)

    run(cmd, timeout=900)

    if not import_ok(module):
        raise RuntimeError(
            f"{package} was installed but '{module}' still cannot import."
        )

    mark_changed()
    print(f"✓ {package}: import OK")

    return True


def repo_is_valid(path):
    """
    Determine whether a repository is actually usable rather than merely
    having a directory with the correct name.
    """

    checks = {
        OPENVLA: [
            f"{OPENVLA}/prismatic",
        ],
        LIBERO: [
            f"{LIBERO}/libero",
            f"{LIBERO}/libero/libero",
        ],
        ARMAGENT: [
            f"{ARMAGENT}/arm_agent/arm_agent.py",
        ],
        SIMPLER: [
            f"{SIMPLER}/simpler_env",
            f"{SIMPLER}/ManiSkill2_real2sim",
        ],
    }

    if path not in checks:
        return os.path.isdir(path)

    return all(os.path.exists(x) for x in checks[path])

def python_import_test(module):
    r = subprocess.run(
        [
            PYTHON,
            "-c",
            f"import {module}",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={
            **os.environ,
            "MPLBACKEND": "Agg",
            "MUJOCO_GL": "egl",
        },
    )
    return r.returncode == 0



# =============================================================================
# 1. MOUNT DRIVE
# =============================================================================

print("=" * 72)
print("MASTER PERSISTENT OPENVLA + ARMAGENT SETUP")
print("=" * 72)

print("\n[1] Mounting Google Drive...")

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)

print("✓ Drive mounted")
print("✓ Persistent root:", DRIVE_ROOT)
print("✓ HF cache:", HF_CACHE)


# =============================================================================
# 2. SYSTEM DEPENDENCIES
# =============================================================================

print("\n[2] Checking system dependencies...")

SYSTEM_DEPENDENCIES = {
    "git": "git",
    "wget": "wget",
    "curl": "curl",
    "zstd": "zstd",
    "ffmpeg": "ffmpeg",
    "vulkaninfo": "vulkan-tools",
}

missing_commands = [
    command
    for command in SYSTEM_DEPENDENCIES
    if shutil.which(command) is None
]

if missing_commands:

    missing_packages = [
        SYSTEM_DEPENDENCIES[command]
        for command in missing_commands
    ]

    print("Missing system commands:", ", ".join(missing_commands))
    print("Installing only missing packages:", ", ".join(missing_packages))

    run(
        [
            "bash",
            "-lc",
            "apt-get update -qq && "
            "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
            + " ".join(missing_packages),
        ],
        timeout=600,
    )

else:
    print("✓ All required system commands already available — SKIP")


# =============================================================================
# 3. PREPARE PYTHON ENVIRONMENT
# =============================================================================

print("\n[3] Preparing Python environment...")

if os.path.isdir(ENV):

    print(f"✓ Existing environment found: {ENV}")
    print("→ KEEPING existing environment")
    print("→ NO environment restoration")
    print("→ NO environment recreation")

elif os.path.isfile(ENV_ARCHIVE):

    print("✓ Environment archive found:")
    print(" ", ENV_ARCHIVE)

    print("→ Local environment missing")
    print("→ Restoring environment from Drive...")

    run(
        [
            "bash",
            "-lc",
            f"tar --zstd -xf '{ENV_ARCHIVE}' -C /content",
        ],
        timeout=1800,
    )

    if not os.path.isdir(ENV):
        raise RuntimeError(
            "Environment archive was extracted, but "
            f"{ENV} was not created."
        )

    print("✓ Environment restored")

else:

    print("⚠ No environment and no archive.")
    print("→ Creating a fresh Python 3.11 environment.")

    installer = "/content/miniconda.sh"

    if not os.path.isdir(MINICONDA):

        if os.path.exists(installer):
            os.remove(installer)

        run(
            [
                "wget",
                "-q",
                "https://repo.anaconda.com/miniconda/"
                "Miniconda3-latest-Linux-x86_64.sh",
                "-O",
                installer,
            ],
            timeout=600,
        )

        run(
            [
                "bash",
                installer,
                "-b",
                "-p",
                MINICONDA,
            ],
            timeout=600,
        )

    else:
        print("✓ Miniconda already exists — SKIP")

    if os.path.exists(ENV):
        shutil.rmtree(ENV)

    run(
        [
            CONDA,
            "create",
            "-y",
            "-p",
            ENV,
            "-c",
            "conda-forge",
            "python=3.11",
            "pip",
        ],
        timeout=1800,
        check=False,
    )

    # If the first solver attempt failed, retry classic solver.
    if not os.path.isdir(ENV):
        run(
            [
                CONDA,
                "create",
                "-y",
                "-p",
                ENV,
                "-c",
                "conda-forge",
                "--solver=classic",
                "python=3.11",
                "pip",
            ],
            timeout=1800,
        )

    mark_changed()

if not os.path.exists(PYTHON):
    raise RuntimeError(
        f"Python executable not found: {PYTHON}"
    )

run([PYTHON, "--version"], env=VERIFY_ENV, timeout=30)


# =============================================================================
# 4. REPOSITORIES
# =============================================================================

print("\n[4] Preparing repositories...")

required_repos = [
    OPENVLA,
    LIBERO,
    ARMAGENT,
    SIMPLER,
]

all_repos_valid = all(repo_is_valid(path) for path in required_repos)

if all_repos_valid:

    print("✓ All required repositories already present")
    print("→ KEEPING repositories")
    print("→ NO repository restoration")

elif os.path.isfile(REPO_ARCHIVE):

    print("✓ Repository archive found:")
    print(" ", REPO_ARCHIVE)

    print("→ One or more repositories are missing/incomplete")
    print("→ Restoring repository archive...")

    # Only remove repositories that are incomplete.
    for path in required_repos:
        if not repo_is_valid(path) and os.path.exists(path):
            print("Removing incomplete repository:", path)
            shutil.rmtree(path)

    run(
        [
            "bash",
            "-lc",
            f"tar --zstd -xf '{REPO_ARCHIVE}' -C /content",
        ],
        timeout=900,
    )

    missing = [
        path for path in required_repos
        if not repo_is_valid(path)
    ]

    if missing:
        raise RuntimeError(
            "Repository restoration incomplete:\n"
            + "\n".join(missing)
        )

    mark_changed()
    print("✓ Repositories restored")

else:

    print("⚠ No repository archive.")
    print("→ Cloning missing repositories...")

    if not repo_is_valid(OPENVLA):
        run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/openvla/openvla.git",
                OPENVLA,
            ],
            timeout=900,
        )
        mark_changed()

    if not repo_is_valid(LIBERO):
        run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/Lifelong-Robot-Learning/LIBERO.git",
                LIBERO,
            ],
            timeout=900,
        )
        mark_changed()

    if not repo_is_valid(ARMAGENT):
        run(
            [
                "git",
                "clone",
                "https://github.com/vnitinreddy/panda_pick",
                "--depth",
                "1",
                ARMAGENT,
            ],
            timeout=900,
        )
        mark_changed()

    if not repo_is_valid(SIMPLER):
        run(
            [
                "git",
                "clone",
                "--recurse-submodules",
                "--depth",
                "1",
                "https://github.com/simpler-env/SimplerEnv",
                SIMPLER,
            ],
            timeout=1200,
        )
        mark_changed()


# =============================================================================
# 5. BASIC PYTHON BUILD TOOLS
# =============================================================================

print("\n[5] Checking Python build tools...")

# These are lightweight and safe to install only if absent.
install_if_missing("packaging", "packaging")
install_if_missing("wheel", "wheel")
install_if_missing("ninja", "ninja")
install_if_missing("cmake", "cmake")


# =============================================================================
# 6. CRITICAL VERSION PINS
# =============================================================================

print("\n[6] Checking critical compatibility pins...")

print("\nNumPy")
ensure_version("numpy", NUMPY_VERSION)

print("\nprotobuf")
ensure_version("protobuf", PROTOBUF_VERSION)

print("\nTensorFlow Metadata")
ensure_version("tensorflow-metadata", TFMD_VERSION)

print("\nHugging Face Hub")
ensure_version("huggingface-hub", HF_HUB_VERSION)

print("\nsetuptools")
ensure_version("setuptools", SETUPTOOLS_VERSION)


# =============================================================================
# 7. TENSORFLOW STACK
# =============================================================================

print("\n[7] Checking TensorFlow stack...")

# Install only if missing.
install_if_missing(
    "tensorflow==2.15.0",
    "tensorflow",
)

install_if_missing(
    "tensorflow-datasets==4.9.3",
    "tensorflow_datasets",
)

# TFMD was pinned above.

TF_CHECK = r"""
import importlib.metadata as md
import numpy
import google.protobuf
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_metadata

print("NumPy              :", numpy.__version__)
print("protobuf           :", google.protobuf.__version__)
print("TensorFlow         :", tf.__version__)
print("TFDS               :", tfds.__version__)
print("TensorFlow Metadata:", md.version("tensorflow-metadata"))

assert numpy.__version__ == "1.26.4"
assert google.protobuf.__version__ == "4.25.9"
assert tf.__version__ == "2.15.0"
assert tfds.__version__ == "4.9.3"
assert md.version("tensorflow-metadata") == "1.15.0"

from tensorflow_metadata.proto.v0 import schema_pb2

print("schema_pb2         : OK")
print("=== TF STACK OK ===")
"""

run(
    [PYTHON, "-c", TF_CHECK],
    env=VERIFY_ENV,
    timeout=180,
)

# =============================================================================
# 8. OPENVLA / Prismatic
# =============================================================================

print("\n[8] Checking OpenVLA / prismatic...")

if import_ok("prismatic"):

    print("✓ OpenVLA / prismatic already imports — SKIP install")

else:

    print("→ OpenVLA / prismatic is missing")
    print("→ Installing OpenVLA editable from local repository...")

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "-e",
            OPENVLA,
        ],
        env=VERIFY_ENV,
        timeout=1200,
    )

    mark_changed()

    if not import_ok("prismatic"):
        raise RuntimeError(
            "OpenVLA installation completed, but "
            "'prismatic' still cannot be imported."
        )

    print("✓ OpenVLA / prismatic installed successfully")

# =============================================================================
# 9. Override PYTORCH / CUDA
# =============================================================================

print("\n[9] Checking PyTorch / CUDA...")

TORCH_INFO = r"""
import torch

print("PyTorch version :", torch.__version__)
print("Torch CUDA      :", torch.version.cuda)
print("CUDA available  :", torch.cuda.is_available())
print("GPU count       :", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("Device          :", torch.cuda.current_device())
"""

run(
    [PYTHON, "-c", TORCH_INFO],
    env=VERIFY_ENV,
    timeout=90,
)

torch_version = package_version("torch")

print("\nDetected PyTorch:", torch_version)


# -------------------------------------------------------------------------
# Known-good OpenVLA / ArmAgent stack
#
# This is the configuration that successfully ran inference:
#
#   PyTorch       2.11.0
#   Torch CUDA    12.8
#   CUDA available
#   A100 GPU
#
# IMPORTANT:
#   Do NOT reinstall if the correct stack is already present.
#   Only repair PyTorch when the installed version is wrong.
# -------------------------------------------------------------------------
'''
TORCH_NEEDS_REPAIR = False

if torch_version is None:
    print("→ PyTorch is not installed.")
    TORCH_NEEDS_REPAIR = True

elif not torch_version.startswith("2.11.0"):
    print(
        f"→ PyTorch {torch_version} is not the known-good 2.11.0 stack."
    )
    TORCH_NEEDS_REPAIR = True

else:
    # Version is correct, but also verify that this is the CUDA 12.8 build.
    TORCH_CUDA_CHECK = r"""
import torch

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)

if torch.version.cuda != "12.8":
    raise SystemExit(1)
"""

    cuda_check = run(
        [PYTHON, "-c", TORCH_CUDA_CHECK],
        env=VERIFY_ENV,
        timeout=90,
        check=False,
    )

    if cuda_check.returncode == 0:
        print("✓ PyTorch 2.11.0 + CUDA 12.8 already installed — SKIP reinstall")
    else:
        print(
            "→ PyTorch 2.11.0 is installed, but it is not the CUDA 12.8 build."
        )
        TORCH_NEEDS_REPAIR = True

# -------------------------------------------------------------------------
# Install / repair only when necessary
# -------------------------------------------------------------------------

if TORCH_NEEDS_REPAIR:

    print()
    print("→ Installing known-good PyTorch stack:")
    print("    torch       == 2.11.0")
    print("    torchvision")
    print("    torchaudio")
    print("    CUDA runtime = 12.8")
    print()

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--index-url",
            "https://download.pytorch.org/whl/cu128",
            "torch==2.11.0",
            "torchvision",
            "torchaudio",
        ],
        env=VERIFY_ENV,
        timeout=1800,
    )

    mark_changed()

    print("\n✓ PyTorch repair/install complete.")
'''

# -------------------------------------------------------------------------
# Flash Attention
# -------------------------------------------------------------------------

'''
print("\n Checking Flash Attention 2.5.5...")

FLASH_OK = False

try:
    result = subprocess.run(
        [
            PYTHON,
            "-c",
            """
import flash_attn
print("flash_attn:", flash_attn.__version__)

assert flash_attn.__version__ == "2.5.5"

from flash_attn import flash_attn_func
print("flash_attn_func import: OK")
""",
        ],
        env=VERIFY_ENV,
        capture_output=True,
        text=True,
    )

    if result.returncode == 0:
        print(result.stdout)
        FLASH_OK = True
        print("✓ Flash Attention 2.5.5 is working — SKIP")
    else:
        print("⚠ Flash Attention import failed.")
        print(result.stderr[-4000:])

except Exception as e:
    print("⚠ Flash Attention check failed:", e)


if not FLASH_OK:

    print("\nRebuilding Flash Attention 2.5.5 against final PyTorch...")

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "flash-attn",
        ],
        env=VERIFY_ENV,
    )

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "flash-attn==2.5.5",
        ],
        env=VERIFY_ENV,
        timeout=1800,
    )

    # Hard validation after installation.
    run(
        [
            PYTHON,
            "-c",
            """
import torch
import flash_attn

print("PyTorch       :", torch.__version__)
print("Torch CUDA    :", torch.version.cuda)
print("Flash Attention:", flash_attn.__version__)

assert torch.__version__.startswith("2.11.0")
assert torch.version.cuda == "12.8"
assert flash_attn.__version__ == "2.5.5"

from flash_attn import flash_attn_func

print("flash_attn_func import: OK")
print("=== FLASH ATTENTION OK ===")
""",
        ],
        env=VERIFY_ENV,
        timeout=300,
    )

    print("✓ Flash Attention rebuilt successfully.")
'''

if not python_import_test("flash_attn"):

    print("\nFlash Attention missing — installing 2.5.5...")

    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "flash-attn==2.5.5",
        "--no-build-isolation",
    ])

else:

    print("Flash Attention already imports.")



# -------------------------------------------------------------------------
# Final verification
# -------------------------------------------------------------------------

'''
TORCH_VERSION_CHECK = r"""
import torch

print("PyTorch version :", torch.__version__)
print("Torch CUDA      :", torch.version.cuda)
print("CUDA available  :", torch.cuda.is_available())
print("GPU count       :", torch.cuda.device_count())

assert torch.__version__.startswith("2.11.0"), (
    f"Expected PyTorch 2.11.0, found {torch.__version__}"
)

assert torch.version.cuda == "12.8", (
    f"Expected Torch CUDA 12.8, found {torch.version.cuda}"
)

assert torch.cuda.is_available(), (
    "CUDA is not available."
)

assert torch.cuda.device_count() >= 1, (
    "No CUDA GPU detected."
)

print("GPU             :", torch.cuda.get_device_name(0))
print("Device          :", torch.cuda.current_device())

print("=== PYTORCH / CUDA OK ===")
"""

run(
    [PYTHON, "-c", TORCH_VERSION_CHECK],
    env=VERIFY_ENV,
    timeout=120,
)
'''

# -------------------------------------------------------------------------
# Verify
# -------------------------------------------------------------------------

OPENVLA_CHECK = r"""
import prismatic

print("prismatic:", prismatic.__file__)
print("OpenVLA: OK")
"""

run(
    [PYTHON, "-c", OPENVLA_CHECK],
    env=VERIFY_ENV,
    timeout=180,
)


FLASH_CHECK = r"""
import flash_attn

print("Flash Attention:", flash_attn.__version__)

assert flash_attn.__version__ == "2.5.5"

print("Flash Attention: OK")
"""

run(
    [PYTHON, "-c", FLASH_CHECK],
    env=VERIFY_ENV,
    timeout=90,
)

print("=== OPENVLA / FLASH ATTENTION OK ===")


# =============================================================================
# 10. LIBERO PACKAGE LAYOUT
# =============================================================================

print("\n[10] Checking LIBERO package layout...")

outer_init = Path(LIBERO) / "libero" / "__init__.py"

if not outer_init.exists():

    print("→ LIBERO outer package marker missing")
    print("→ Creating:", outer_init)

    outer_init.touch()
    mark_changed()

else:
    print("✓ LIBERO package marker already exists — SKIP")


# Install LIBERO only if it doesn't import.
if not import_ok("libero"):

    print("→ LIBERO import missing — installing editable")

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "-e",
            LIBERO,
            "--no-deps",
        ],
        env=VERIFY_ENV,
        timeout=1200,
    )

    mark_changed()

else:
    print("✓ LIBERO already imports — SKIP")


# =============================================================================
# 11. LIBERO ROBOT REQUIREMENTS
# =============================================================================

print("\n[11] Checking LIBERO robot requirements...")

libero_requirements = (
    Path(OPENVLA)
    / "experiments"
    / "robot"
    / "libero"
    / "libero_requirements.txt"
)

if libero_requirements.exists():

    print("✓ LIBERO robot requirements file found")

    # Install individual requirements only if their imports are absent would
    # require maintaining a package mapping. We therefore do NOT blindly run
    # the entire requirements file on every session.
    #
    # The actual required stack is verified below.

else:

    print("⚠ Warning: LIBERO robot requirements file not found:")
    print(" ", libero_requirements)

# =============================================================================
# 12. MUJOCO
# =============================================================================

print("\n[12] Checking MuJoCo...")

mujoco_current = package_version("mujoco")

print("Installed MuJoCo:", mujoco_current)

if mujoco_current == MUJOCO_VERSION:

    print(
        f"✓ MuJoCo {MUJOCO_VERSION} already installed — SKIP reinstall"
    )

else:

    print(
        f"→ MuJoCo {mujoco_current} is installed."
    )
    print(
        f"→ Replacing it with required MuJoCo {MUJOCO_VERSION}..."
    )

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-deps",
            f"mujoco=={MUJOCO_VERSION}",
        ],
        env=VERIFY_ENV,
        timeout=600,
    )

    mark_changed()

    mujoco_current = package_version("mujoco")

    if mujoco_current != MUJOCO_VERSION:
        raise RuntimeError(
            f"MuJoCo installation failed: "
            f"expected {MUJOCO_VERSION}, found {mujoco_current}"
        )

    print(f"✓ MuJoCo successfully changed to {MUJOCO_VERSION}")


# -------------------------------------------------------------------------
# Verify MuJoCo + robosuite together
# -------------------------------------------------------------------------

MUJOCO_CHECK = r"""
import mujoco
import robosuite

print("MuJoCo    :", mujoco.__version__)
print("robosuite :", robosuite.__version__)

assert mujoco.__version__ == "3.3.1"

print("=== MUJOCO / ROBOSUITE OK ===")
"""

run(
    [PYTHON, "-c", MUJOCO_CHECK],
    env=VERIFY_ENV,
    timeout=90,
)


# =============================================================================
# 13. LIBERO BDDL LAYOUT WORKAROUND
# =============================================================================

print("\n[13] Checking LIBERO BDDL layout...")

BDDL_ROOT = Path(
    LIBERO
) / "libero" / "libero" / "bddl_files"

OBJECT_DIR = BDDL_ROOT / "libero_object"

if not OBJECT_DIR.is_dir():
    raise RuntimeError(
        "LIBERO libero_object BDDL directory not found:\n"
        f"{OBJECT_DIR}"
    )

source_files = sorted(OBJECT_DIR.glob("*.bddl"))

if not source_files:
    raise RuntimeError(
        f"No BDDL files found in {OBJECT_DIR}"
    )

copied = 0
already_present = 0

for src in source_files:

    dst = BDDL_ROOT / src.name

    if dst.exists():
        already_present += 1

    else:
        shutil.copy2(src, dst)
        copied += 1
        mark_changed()

print(f"✓ libero_object BDDL source files: {len(source_files)}")
print(f"✓ Already present at parent:       {already_present}")
print(f"✓ Copied this session:              {copied}")


SALAD_BDDL = (
    BDDL_ROOT
    / "pick_up_the_salad_dressing_and_place_it_in_the_basket.bddl"
)

if not SALAD_BDDL.exists():
    raise RuntimeError(
        "Required salad-dressing BDDL file is missing:\n"
        f"{SALAD_BDDL}"
    )

print("✓ Salad dressing BDDL: READY")


# =============================================================================
# 14. SIMPLERENV / MANISKILL
# =============================================================================

print("\n[14] Checking SimplerEnv / ManiSkill...")

MANISKILL = Path(SIMPLER) / "ManiSkill2_real2sim"

if not MANISKILL.is_dir():
    raise RuntimeError(
        "SimplerEnv exists but ManiSkill2_real2sim is missing:\n"
        f"{MANISKILL}"
    )

if not import_ok("mani_skill2_real2sim"):

    print("→ ManiSkill2_real2sim missing — installing editable")

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "-e",
            str(MANISKILL),
        ],
        env=VERIFY_ENV,
        timeout=1200,
    )

    mark_changed()

else:
    print("✓ ManiSkill2_real2sim already imports — SKIP")


if not import_ok("simpler_env"):

    print("→ SimplerEnv missing — installing editable")

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "-e",
            SIMPLER,
        ],
        env=VERIFY_ENV,
        timeout=1200,
    )

    mark_changed()

else:
    print("✓ SimplerEnv already imports — SKIP")


# =============================================================================
# 15. NUMPY / OPENCV
# =============================================================================

print("\n[15] Checking NumPy / OpenCV...")

# NumPy was already pinned above.
# Do NOT force reinstall.

ensure_version("numpy", NUMPY_VERSION)

opencv_current = package_version("opencv-python-headless")

if opencv_current == OPENCV_VERSION:

    print(
        f"✓ opencv-python-headless=={OPENCV_VERSION} already installed — SKIP"
    )

elif opencv_current is None:

    # Remove conflicting OpenCV package only if necessary.
    run(
        [
            PYTHON,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "opencv-python",
            "opencv-python-headless",
        ],
        env=VERIFY_ENV,
        timeout=300,
        check=False,
    )

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-deps",
            f"opencv-python-headless=={OPENCV_VERSION}",
        ],
        env=VERIFY_ENV,
        timeout=600,
    )

    mark_changed()

else:

    print(
        f"⚠ OpenCV headless version is {opencv_current}; "
        f"required {OPENCV_VERSION}"
    )

    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-deps",
            f"opencv-python-headless=={OPENCV_VERSION}",
        ],
        env=VERIFY_ENV,
        timeout=600,
    )

    mark_changed()


# =============================================================================
# 16. ARMAGENT DEPENDENCIES
# =============================================================================

print("\n[16] Checking ArmAgent dependencies...")

install_if_missing(
    "mbodied",
    "mbodied",
    extra_args=["--no-deps"],
)

install_if_missing(
    "mediapy",
    "mediapy",
    extra_args=["--no-deps"],
)

install_if_missing(
    "jsonref",
    "jsonref",
    extra_args=["--no-deps"],
)

# Gymnasium has a known version in this stack.
ensure_version(
    "gymnasium",
    GYMNASIUM_VERSION,
    no_deps=True,
)

install_if_missing(
    "datasets",
    "datasets",
    extra_args=["--no-deps"],
)

# These are explicitly checked because ArmAgent/mbodied may use them.
install_if_missing(
    "compress-pickle",
    "compress_pickle",
    extra_args=["--no-deps"],
)

install_if_missing(
    "ruamel.yaml",
    "ruamel.yaml",
    extra_args=["--no-deps"],
)


# =============================================================================
# 17. COMPLETE ENVIRONMENT VERIFICATION
# =============================================================================

print("\n[17] COMPLETE ENVIRONMENT VERIFICATION")
print("=" * 72)


'''
    (
        "PyTorch / CUDA",
        r"""
import sys
import torch

print("Python        :", sys.executable)
print("PyTorch       :", torch.__version__)
print("Torch CUDA    :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())

assert sys.executable == "/content/openvla_env/bin/python", (
    f"Wrong Python environment: {sys.executable}"
)

assert torch.__version__.startswith("2.11.0"), (
    f"Expected PyTorch 2.11.0, found {torch.__version__}"
)

assert torch.version.cuda == "12.8", (
    f"Expected Torch CUDA 12.8, found {torch.version.cuda}"
)

assert torch.cuda.is_available(), (
    "CUDA is not available."
)

assert torch.cuda.device_count() >= 1, (
    "No CUDA GPU detected."
)

print("GPU           :", torch.cuda.get_device_name(0))
print("Device        :", torch.cuda.current_device())
print("=== PYTORCH / CUDA OK ===")
""",
        60,
        None,
    ),
'''

CHECKS = [

    (
        "Python",
        r"""
import sys

print(sys.version)

assert sys.version_info.major == 3
assert sys.version_info.minor == 11

print("Python: OK")
""",
        30,
        None,
    ),

    (
        "NumPy",
        r"""
import numpy

print("NumPy:", numpy.__version__)

assert numpy.__version__ == "1.26.4"

print("NumPy: OK")
""",
        30,
        None,
    ),

    (
        "Flash Attention",
        r"""
import flash_attn

print("Flash Attention:", flash_attn.__version__)

assert flash_attn.__version__ == "2.5.5"

print("Flash Attention: OK")
""",
        60,
        None,
    ),

    (
        "TensorFlow",
        r"""
import tensorflow as tf

print("TensorFlow:", tf.__version__)

assert tf.__version__ == "2.15.0"

print("TensorFlow: OK")
""",
        120,
        None,
    ),

    (
        "TensorFlow Datasets",
        r"""
import tensorflow_datasets as tfds

print("TFDS:", tfds.__version__)

assert tfds.__version__ == "4.9.3"

print("TFDS: OK")
""",
        120,
        None,
    ),

    (
        "TensorFlow Metadata",
        r"""
import importlib.metadata as md
import tensorflow_metadata

version = md.version("tensorflow-metadata")

print("TFMD:", version)

assert version == "1.15.0"

from tensorflow_metadata.proto.v0 import schema_pb2

print("schema_pb2: OK")
print("TensorFlow Metadata: OK")
""",
        120,
        None,
    ),

    (
        "protobuf",
        r"""
import google.protobuf

print("protobuf:", google.protobuf.__version__)

assert google.protobuf.__version__ == "4.25.9"

print("protobuf: OK")
""",
        30,
        None,
    ),

    (
        "Hugging Face",
        r"""
import importlib.metadata as md
import transformers
import huggingface_hub

hub = md.version("huggingface-hub")
transformers_version = md.version("transformers")

print("huggingface-hub:", hub)
print("transformers    :", transformers_version)

assert hub == "0.24.7"

print("Hugging Face: OK")
""",
        90,
        None,
    ),

    (
        "OpenVLA",
        r"""
import prismatic

print("prismatic:", prismatic.__file__)
print("OpenVLA: OK")
""",
        180,
        None,
    ),

    (
        "MuJoCo / robosuite",
        r"""
import mujoco
import robosuite

print("MuJoCo:", mujoco.__version__)
print("robosuite:", robosuite.__version__)

assert mujoco.__version__ == "3.3.1"

print("MuJoCo / robosuite: OK")
""",
        90,
        None,
    ),

    (
        "LIBERO",
        r"""
import libero

print("libero:", libero.__file__)

from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv

print("benchmark: OK")
print("get_libero_path: OK")
print("OffScreenRenderEnv: OK")

print("LIBERO: OK")
""",
        120,
        "N\n",
    ),

    (
        "SAPIEN",
        r"""
import pkg_resources
import sapien.core as sapien

print("pkg_resources: OK")
print("SAPIEN: OK")
""",
        120,
        None,
    ),

    (
        "ManiSkill2_real2sim",
        r"""
import mani_skill2_real2sim

print("ManiSkill2_real2sim: OK")
""",
        120,
        None,
    ),

    (
        "SimplerEnv",
        r"""
import simpler_env

print("simpler_env:", simpler_env.__file__)
print("SimplerEnv: OK")
""",
        120,
        None,
    ),

    (
        "mediapy",
        r"""
import mediapy

print("mediapy: OK")
""",
        60,
        None,
    ),

    (
        "mbodied",
        r"""
import mbodied

print("mbodied: OK")

from mbodied.robots import Robot
from mbodied.types.motion.control import HandControl

print("Robot: OK")
print("HandControl: OK")
print("mbodied: OK")
""",
        120,
        None,
    ),

    (
        "ArmAgent files",
        r"""
import os

files = [
    "/content/panda_pick/arm_agent/arm_agent.py",
]

for path in files:
    assert os.path.exists(path), f"Missing: {path}"
    print(os.path.basename(path), ": OK")

print("ArmAgent files: OK")
""",
        30,
        None,
    ),
]


passed = []

for name, code, timeout, input_data in CHECKS:

    print("\n" + "=" * 72)
    print("[CHECK]", name)
    print("=" * 72)

    r = subprocess.run(
        [
            PYTHON,
            "-c",
            code,
        ],
        input=input_data,
        env=VERIFY_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )

    print(r.stdout)

    if r.returncode != 0:
        raise RuntimeError(
            f"{name} verification FAILED."
        )

    passed.append(name)
    print(f"✓ {name} PASSED")


# =============================================================================
# 18. ACTUAL LIBERO ENVIRONMENT CREATION TEST
# =============================================================================

print("\n[18] ACTUAL LIBERO ENVIRONMENT TEST")
print("=" * 72)

LIBERO_ENV_TEST = r"""
import os

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

print("Creating LIBERO environment...")

suite = benchmark.get_benchmark("libero_object")(0)

task = suite.get_task(2)

print("Task:", task.language)
print("BDDL:", task.bddl_file)

bddl_root = get_libero_path("bddl_files")

bddl_file = os.path.join(
    bddl_root,
    task.bddl_file,
)

print("Resolved BDDL:", bddl_file)

assert os.path.exists(
    bddl_file
), f"BDDL file missing: {bddl_file}"

env_args = {
    "bddl_file_name": bddl_file,
    "camera_heights": 256,
    "camera_widths": 256,
}

env = OffScreenRenderEnv(**env_args)

print("LIBERO environment created successfully!")

env.close()

print("=== LIBERO ENVIRONMENT SUCCESS ===")
"""

run(
    [
        PYTHON,
        "-c",
        LIBERO_ENV_TEST,
    ],
    env=VERIFY_ENV,
    input_data="N\n",
    timeout=180,
)


# =============================================================================
# 19. FINAL GPU TEST
# =============================================================================

print("\n[19] FINAL GPU TEST")
print("=" * 72)

GPU_TEST = r"""
import torch

assert torch.cuda.is_available()

device = torch.device("cuda:0")

x = torch.randn(
    (1024, 1024),
    device=device,
    dtype=torch.bfloat16,
)

y = x @ x

torch.cuda.synchronize()

print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)
print("Tensor dtype:", x.dtype)
print("Tensor device:", x.device)
print("GPU computation: OK")
"""

run(
    [PYTHON, "-c", GPU_TEST],
    env=VERIFY_ENV,
    timeout=90,
)


# =============================================================================
# 20. REFRESH ARCHIVES ONLY WHEN NECESSARY
# =============================================================================
# =============================================================================
# 20. REFRESH ARCHIVES ONLY WHEN NECESSARY
# =============================================================================

print("\n[20] Persistence")

# FORCE_REFRESH_ARCHIVES is an explicit override.
# Otherwise, archives are refreshed only when an actual setup mutation
# called mark_changed() earlier in this cell.

refresh_archives = (
    FORCE_REFRESH_ARCHIVES
    or (REFRESH_ARCHIVES_IF_CHANGED and changed)
)

if not refresh_archives:

    print("✓ No environment/repository changes detected.")
    print("✓ Existing persistent archives are already current.")
    print("→ Skipping archive creation.")

else:

    print("→ Environment/repository changes detected.")
    print("→ Refreshing persistent archives...")

    # -------------------------------------------------------------------------
    # Repository archive
    # -------------------------------------------------------------------------

    print("\nCreating repository archive...")

    tmp_repo_archive = REPO_ARCHIVE + ".tmp"

    if os.path.exists(tmp_repo_archive):
        os.remove(tmp_repo_archive)

    run(
        [
            "bash",
            "-lc",
            f"""
tar --zstd -cf '{tmp_repo_archive}' \
    -C /content \
    openvla \
    LIBERO \
    panda_pick/arm_agent \
    SimplerEnv
            """,
        ],
        timeout=1200,
    )

    os.replace(tmp_repo_archive, REPO_ARCHIVE)

    print("✓ Repository archive refreshed.")

    # -------------------------------------------------------------------------
    # Python environment archive
    # -------------------------------------------------------------------------

    print("\nCreating Python environment archive...")

    tmp_env_archive = ENV_ARCHIVE + ".tmp"

    if os.path.exists(tmp_env_archive):
        os.remove(tmp_env_archive)

    run(
        [
            "bash",
            "-lc",
            f"""
tar --zstd -cf '{tmp_env_archive}' \
    -C /content \
    openvla_env
            """,
        ],
        timeout=1800,
    )

    os.replace(tmp_env_archive, ENV_ARCHIVE)

    print("✓ Python environment archive refreshed.")

    print("\n✓ Persistent archives refreshed successfully.")

'''
print("\n[20] Persistence")

if FORCE_REFRESH_ARCHIVES:
    refresh_archives = True
elif REFRESH_ARCHIVES_IF_CHANGED and changed:
    refresh_archives = True
else:
    refresh_archives = False


if refresh_archives:

    print("→ Setup changed the environment/repositories.")
    print("→ Refreshing persistent archives...")

    # -------------------------------------------------------------------------
    # Repository archive
    # -------------------------------------------------------------------------

    print("\nCreating repository archive...")

    tmp_repo_archive = REPO_ARCHIVE + ".tmp"

    if os.path.exists(tmp_repo_archive):
        os.remove(tmp_repo_archive)

    run(
        [
            "bash",
            "-lc",
            f"""
tar --zstd -cf '{tmp_repo_archive}' \
    -C /content \
    openvla \
    LIBERO \
    panda_pick\arm_agent \
    SimplerEnv
            """,
        ],
        timeout=1200,
    )

    os.replace(tmp_repo_archive, REPO_ARCHIVE)

    # -------------------------------------------------------------------------
    # Environment archive
    # -------------------------------------------------------------------------

    print("\nCreating Python environment archive...")

    tmp_env_archive = ENV_ARCHIVE + ".tmp"

    if os.path.exists(tmp_env_archive):
        os.remove(tmp_env_archive)

    run(
        [
            "bash",
            "-lc",
            f"""
tar --zstd -cf '{tmp_env_archive}' \
    -C /content \
    openvla_env
            """,
        ],
        timeout=1800,
    )

    os.replace(tmp_env_archive, ENV_ARCHIVE)

    print("\n✓ Persistent archives refreshed")

else:

    print("✓ No setup changes detected")
    print("✓ Persistent archives left untouched")
'''

# =============================================================================
# 21. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 72)
print("SETUP COMPLETE — READY FOR ARMAGENT INFERENCE")
print("=" * 72)

print(
    f"""
Python environment:
    {ENV}

Python:
    {PYTHON}

OpenVLA:
    {OPENVLA}

LIBERO:
    {LIBERO}

ArmAgent:
    {ARMAGENT}

SimplerEnv:
    {SIMPLER}

Hugging Face cache:
    {HF_CACHE}

Environment archive:
    {ENV_ARCHIVE}

Repository archive:
    {REPO_ARCHIVE}

--------------------------------------------------------

Verified:
    ✓ Python 3.11
    ✓ PyTorch 2.11.x
    ✓ CUDA 12.8
    ✓ A100 CUDA execution
    ✓ Flash Attention 2.5.5
    ✓ NumPy 1.26.4
    ✓ TensorFlow 2.15.0
    ✓ TFDS 4.9.3
    ✓ TensorFlow Metadata 1.15.0
    ✓ protobuf 4.25.9
    ✓ huggingface-hub 0.24.7
    ✓ MuJoCo 3.3.1
    ✓ robosuite
    ✓ OpenVLA
    ✓ LIBERO
    ✓ LIBERO BDDL layout
    ✓ LIBERO OffScreenRenderEnv
    ✓ SAPIEN
    ✓ ManiSkill2_real2sim
    ✓ SimplerEnv
    ✓ mediapy
    ✓ mbodied
    ✓ ArmAgent files
    ✓ GPU computation

Setup changes this session:
    {changed}

READY FOR INFERENCE.
"""
)

MASTER PERSISTENT OPENVLA + ARMAGENT SETUP

[1] Mounting Google Drive...
Mounted at /content/drive
✓ Drive mounted
✓ Persistent root: /content/drive/MyDrive/OpenVLA
✓ HF cache: /content/drive/MyDrive/OpenVLA/huggingface_cache

[2] Checking system dependencies...
Missing system commands: zstd, vulkaninfo
Installing only missing packages: zstd, vulkan-tools

$ bash -lc apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd vulkan-tools
------------------------------------------------------------------------
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libvulkan1:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading dat

In [2]:
#@title inference
import subprocess

cmd = [
    "/content/openvla_env/bin/python",
    "/content/panda_pick/arm_agent/arm_agent.py",
    "--prompt", "pick up the salad dressing and place it in the basket",
    "--task", "libero_object",
    "--task_id", "2",
    "--image_resize", "1024",
    "--output_video", "/content/panda_pick/arm_agent/outputs/videos",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("ArmAgent PID:", p.pid)

# Print output as it arrives
while True:
    line = p.stdout.readline()

    if line:
        print(line, end="", flush=True)

    elif p.poll() is not None:
        break

print("\nArmAgent exited with:", p.returncode)

from IPython.display import Video, display
from pathlib import Path

video_dir = Path("/content/panda_pick/arm_agent/outputs/videos")

videos = sorted(
    video_dir.glob("*.mp4"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not videos:
    print("No MP4 video found.")
else:
    video = videos[0]
    print("Playing:", video)
    display(Video(str(video), embed=True))

ArmAgent PID: 9048
2026-09-08 03:41:56.011594: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 03:41:56.061583: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-08 03:41:56.061636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-08 03:41:56.063005: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-08 03:41:56.070747: I tensorflow/core/platfo